In [ ]:
import numpy as np
import sklearn.decomposition as dp
import matplotlib.pyplot as plt 
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf
import importlib

sys.path.append('/home/austin/Keller_Lab/Depression_EEG_Biomarker/Code/NMF_SAE')
from nmf_joint_vae2 import dNMF_IS_vae


gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus: 
  # Restrict TensorFlow to only allocate 1GB of memory on the first GPU
  try:
    tf.config.experimental.set_virtual_device_configuration(
        gpus[0],
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=8192)])
    logical_gpus = tf.config.experimental.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e) 

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)


In [ ]:
import nmf_joint_vae2

In [ ]:
importlib.reload(nmf_joint_vae2)

In [ ]:
indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))
X = X[indx_tot]

print(mouse.shape)
print(X.shape)

X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]



In [ ]:
power,coherence,granger,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X_new = np.hstack((power,coherence,granger))

windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])

idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


In [ ]:
mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]


ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1

print('>>>>>>>>>>>>>.')
print(y_new.shape)
print(ids.shape)
print(mouse_new.shape)
print(X_new.shape)

X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]

print(X_train_new.shape)
print(y_train_new.shape)
print(X_test_new.shape)
print(y_test_new.shape)

#>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


In [ ]:
X_train_tot = np.vstack((X_train,X_train_new))
y_train_tot = np.concatenate((y_train,y_train_new))
weights_g = np.ones(X_train_tot.shape[0])
weights_s = np.ones(X_train_tot.shape[0])

nFact = 5
myDict = {}

training_pdict = {'batch_size':128,'num_iterations':20000}
reg_pdict = {'reg_features':10.0}


In [ ]:
model = nmf_joint_vae2.dNMF_IS_vae(nFact,supervision_strength=1.0,reg_pdict=reg_pdict,
                        training_pdict=training_pdict,lamb=.1)
model.fit(X_train_tot,y_train_tot)
S_train = model.transform(X_train)
S_test = model.transform(X_test)
S_train_new = model.transform(X_train_new)
S_test_new = model.transform(X_test_new)
components_ = model.W_


In [ ]:
#loss_recon = tf.reduce_mean(tf.abs(X_batch-X_recon))


In [ ]:

myDict['S_train'] = S_train
myDict['S_test'] = S_test
myDict['S_train_new'] = S_train_new
myDict['S_test_new'] = S_test_new
myDict['components'] = components_
myDict['phi'] = model.Phi_

Ex = np.mean(X_train,axis=0)
reconRand_tr = np.mean((X_train-Ex)**2)
reconRand_te = np.mean((X_test-Ex)**2)
print('Random Reconstruction',np.mean((X_train-Ex)**2))
print('Random Reconstruction',np.mean((X_test-Ex)**2))
myDict['recon_random_train'] = reconRand_tr
myDict['recon_random_test'] = reconRand_te
myDict['Ex'] = Ex

X_recon_tr = np.dot(S_train,components_)
X_recon_te = np.dot(S_test,components_)
print('Recon sNMF',np.mean((X_train-X_recon_tr)**2))
print('Recon sNMF',np.mean((X_test-X_recon_te)**2))
recon_snmf_tr = np.mean((X_recon_tr-X_train)**2)
recon_snmf_te = np.mean((X_recon_te-X_test)**2)

myDict['recon_sae_train'] = recon_snmf_tr
myDict['recon_sae_test'] = recon_snmf_te

mice_test = np.unique(m_test)


In [ ]:
print('MSE',np.mean((X_train)**2))
print('MSE',np.mean((X_test)**2))

In [ ]:
plt.plot(model.losses_supervision[-1000:])
plt.show()


In [ ]:
start = 1500
ll_r = model.losses_recon
plt.plot((ll_r[start:-3]+ll_r[start+1:-2]+ll_r[start+2:-1]+ll_r[start+3:])/4)
plt.show()

In [ ]:
Str0 = S_train[:,0]*-1
Ste0 = S_test[:,0]*-1

Str1 = S_train[:,1]
Ste1 = S_test[:,1]

y_df_train = model.decision_function(X_train)
y_df_test = model.decision_function(X_test)

print('First components')
print('Training ROC 0',roc_auc_score(y_train,Str0))
print('Testing ROC 0',roc_auc_score(y_test,Ste0))
print('>>>>>>>>>>>>>>>>')
print('Training ROC 1',roc_auc_score(y_train,Str1))
print('Testing ROC 1',roc_auc_score(y_test,Ste1))

myDict['roc_train0'] = roc_auc_score(y_train,Str0)
myDict['roc_test0'] = roc_auc_score(y_test,Ste0)

myDict['roc_train1'] = roc_auc_score(y_train,Str1)
myDict['roc_test1'] = roc_auc_score(y_test,Ste1)
myDict['model'] = model

idx1 = mice_test[0]==m_test
idx2 = mice_test[1]==m_test
idx3 = mice_test[2]==m_test
print('ROC1',roc_auc_score(y_test[idx1],Ste0[idx1]))
print('ROC2',roc_auc_score(y_test[idx2],Ste0[idx2]))
print('ROC3',roc_auc_score(y_test[idx3],Ste0[idx3]))
print('>>>>>>>>>>>>>>>>')
print('ROC1',roc_auc_score(y_test[idx1],Ste1[idx1]))
print('ROC2',roc_auc_score(y_test[idx2],Ste1[idx2]))
print('ROC3',roc_auc_score(y_test[idx3],Ste1[idx3]))
print('>>>>>>>>>>>>>>>>')
print('ROC1',roc_auc_score(y_test[idx1],y_df_test[idx1]))
print('ROC2',roc_auc_score(y_test[idx2],y_df_test[idx2]))
print('ROC3',roc_auc_score(y_test[idx3],y_df_test[idx3]))


In [ ]:
pickle.dump(myDict,open('Trial2.p','wb'))